# 07 — Custom CNN Training

Train a compact CNN (<50K params) on log-mel spectrogram inputs from Notebook 04 to classify angle_grinder vs. background vs. tools. Implements file-level stratified splits, SpecAugment during training, Adam optimizer, early stopping, LR reduction, model checkpointing, and reports per-class metrics (with emphasis on grinder recall) per Training Plan/Code.


In [6]:
# Imports and setup
from pathlib import Path
import yaml, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
CFG_PATH = PROJECT_ROOT / 'config.yaml'
NPY_DIR = PROJECT_ROOT / 'data' / 'processed' / 'neural_features'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
MODEL_DIR = PROJECT_ROOT / 'models' / 'neural'
for d in [FIG_DIR, METRICS_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Using TensorFlow:', tf.__version__)


Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv4
Using TensorFlow: 2.17.1


In [7]:
# Load config and data
with open(CFG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)
audio_cfg = cfg.get('audio', {})
TARGET_SR = int(audio_cfg.get('sample_rate', 16000))
N_MELS = int(audio_cfg.get('n_mels', 40))
TARGET_T = int(audio_cfg.get('target_frames', 99))

# Load arrays from 04 (create both 1ch and 3ch variants if available)
X1_path = NPY_DIR / 'spectrograms_1ch.npy'
X3_path = NPY_DIR / 'spectrograms_3ch.npy'
y_path  = NPY_DIR / 'labels.npy'
X = None
if X3_path.exists():
    X = np.load(X3_path)   # (N, T, M, 3)
    input_channels = 3
elif X1_path.exists():
    X = np.load(X1_path)   # (N, T, M, 1)
    input_channels = 1
else:
    raise FileNotFoundError('No neural features found. Run Notebook 04.')

y_str = np.load(y_path)  # Load as strings (e.g., 'angle_grinder')

# Encode labels to integers (0, 1, 2) using LabelEncoder from classical training
# Or create a new encoder if not available
le_path = PROJECT_ROOT / 'models' / 'classical' / 'label_encoder.pkl'
if le_path.exists():
    # Reuse encoder from Notebook 05 for consistency
    import joblib
    le = joblib.load(le_path)
    print('Loaded LabelEncoder from Notebook 05:', le.classes_)
else:
    # Create encoder from scratch (if running CNN before classical)
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    le.fit(y_str)
    print('Created new LabelEncoder:', le.classes_)

# Transform labels to integers
y = le.transform(y_str)
classes = le.classes_
num_classes = len(classes)

print('Data:', X.shape, 'Labels:', y.shape, 'Classes:', num_classes, 'channels:', input_channels)
print('Label distribution:', {cls: int(np.sum(y == i)) for i, cls in enumerate(classes)})

# File-level split: best effort from manifest (fallback to stratified sample-level)
manifest_path = METRICS_DIR / 'neural_features_manifest.csv'
if manifest_path.exists():
    man = pd.read_csv(manifest_path)
    if 'original_path' in man.columns:
        # Map each sample index to its file stem (assumes order kept from 04)
        stems = man['original_path'].apply(lambda p: Path(str(p)).stem).values[:len(y)]
        df_idx = pd.DataFrame({'idx': np.arange(len(y)), 'y': y, 'stem': stems})
        # Group by stem for file-level splits
        unique_stems = df_idx[['stem', 'y']].drop_duplicates()
        strat_train, strat_test = train_test_split(unique_stems, test_size=0.2, stratify=unique_stems['y'], random_state=42)
        # Map back to indices
        train_stems = set(strat_train['stem'])
        test_stems  = set(strat_test['stem'])
        train_idx = df_idx['idx'][df_idx['stem'].isin(train_stems)].values
        test_idx  = df_idx['idx'][df_idx['stem'].isin(test_stems)].values
        # Further split train into train/val
        X_train, X_val, y_train, y_val = train_test_split(X[train_idx], y[train_idx], test_size=0.2, stratify=y[train_idx], random_state=42)
        X_test, y_test = X[test_idx], y[test_idx]
    else:
        # Fallback to stratified sample-level split
        X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
        X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)
else:
    # Sample-level fallback
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print('Splits:', X_train.shape, X_val.shape, X_test.shape)
input_shape = X_train.shape[1:]
class_count = np.bincount(y_train, minlength=num_classes)
class_weight = {i: float(class_count.max() / c) if c > 0 else 0.0 for i, c in enumerate(class_count)}
print('Class weight:', class_weight)

Loaded LabelEncoder from Notebook 05: ['angle_grinder' 'background' 'tools']
Data: (19044, 99, 40, 3) Labels: (19044,) Classes: 3 channels: 3
Label distribution: {'angle_grinder': 6378, 'background': 8069, 'tools': 4597}
Splits: (15235, 99, 40, 3) (1904, 99, 40, 3) (1905, 99, 40, 3)
Class weight: {0: 1.2651901215209722, 1: 1.0, 2: 1.7550299075584557}


In [8]:
# SpecAugment (time/frequency masking) layer as Lambda
def spec_augment(x, max_time_mask=8, max_freq_mask=8):
    # x: (T, M, C) in batch
    t = tf.shape(x)[1]; m = tf.shape(x)[2]
    # Time mask
    t0 = tf.random.uniform([], 0, t - max_time_mask, dtype=tf.int32)
    dt = tf.random.uniform([], 0, max_time_mask, dtype=tf.int32)
    mask_t = tf.concat([tf.ones((t0, m, 1)), tf.zeros((dt, m, 1)), tf.ones((t - t0 - dt, m, 1))], axis=0)
    # Freq mask
    f0 = tf.random.uniform([], 0, m - max_freq_mask, dtype=tf.int32)
    df = tf.random.uniform([], 0, max_freq_mask, dtype=tf.int32)
    mask_f = tf.concat([tf.ones((t, f0, 1)), tf.zeros((t, df, 1)), tf.ones((t, m - f0 - df, 1))], axis=1)
    return x * mask_t * mask_f

def make_cnn(input_shape, num_classes, channels=1, dropout=0.3):
    inputs = keras.Input(shape=input_shape)  # (T, M, C)
    x = layers.Lambda(lambda z: spec_augment(z))(inputs)
    x = layers.Conv2D(16, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs, name='bikeai_cnn')
    return model

model = make_cnn(input_shape, num_classes, channels=input_channels, dropout=0.4)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()
# Params check (<50k)
print('Total params:', model.count_params())


2025-11-15 00:26:52.398164: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-11-15 00:26:52.398379: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-11-15 00:26:52.399118: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-11-15 00:26:52.399274: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-15 00:26:52.399438: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "bikeai_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 99, 40, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 99, 40, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 99, 40, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 99, 40, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 20, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 49, 20, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 49, 20, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 24, 10, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 10, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 10, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 5, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 5, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3840)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       491,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 516,067 (1.97 MB)

 Trainable params: 515,843 (1.97 MB)

 Non-trainable params: 224 (896.00 B)

Total params: 516067


In [9]:
# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-5),
    keras.callbacks.ModelCheckpoint(filepath=MODEL_DIR / 'custom_cnn_best.keras', monitor='val_accuracy', save_best_only=True)
]

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30, batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

# Save final
model.save(MODEL_DIR / 'custom_cnn_final.keras')
pd.DataFrame(history.history).to_csv(METRICS_DIR / 'custom_cnn_history.csv', index=False)
print('Saved best and final models.')


Epoch 1/30


2025-11-15 00:26:53.960816: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


239/239 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.6932 - loss: 4.5352 - val_accuracy: 0.8403 - val_loss: 0.8331 - learning_rate: 0.0010
Epoch 2/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - accuracy: 0.8108 - loss: 2.7175 - val_accuracy: 0.8640 - val_loss: 1.1975 - learning_rate: 0.0010
Epoch 3/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.8419 - loss: 1.7793 - val_accuracy: 0.9128 - val_loss: 0.4635 - learning_rate: 0.0010
Epoch 4/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.8704 - loss: 0.9421 - val_accuracy: 0.8971 - val_loss: 0.4083 - learning_rate: 0.0010
Epoch 5/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.8884 - loss: 0.5292 - val_accuracy: 0.9223 - val_loss: 0.2638 - learning_rate: 0.0010
Epoch 6/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.8993 - loss: 0.4218 - val_accuracy: 0.9086 - val_loss: 0.2687 - learning_rate: 0.0010
Epoch 7/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - accuracy: 0.9050 - loss: 0.4093 - val

In [10]:
# Evaluation
y_pred = model.predict(X_test).argmax(axis=1)
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Custom CNN — Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Pred')
plt.tight_layout()
plt.savefig(FIG_DIR / 'cm_custom_cnn.png', dpi=150)
plt.close()
print('Saved confusion matrix.')


60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       638
           1       0.98      0.99      0.98       807
           2       0.99      0.97      0.98       460

    accuracy                           0.98      1905
   macro avg       0.98      0.98      0.98      1905
weighted avg       0.98      0.98      0.98      1905

Saved confusion matrix.
